In [1]:
import os
import numpy as np
import cv2

from matplotlib import pyplot as plt
import matplotlib 

# Local descriptors
from skimage.feature import hog
from skimage import  exposure
from skimage import feature

from time import time
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import MinMaxScaler

In [2]:
import cv2
import numpy as np
from deepface import DeepFace

def apply_emotion_filter(frame, emotion):
    """Aplica un filtro sencillo según la emoción detectada."""
    emotion = emotion.lower()

    if emotion == "happy":
        # Aumentar saturación (más color)
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        hsv[..., 1] = np.clip(hsv[..., 1] * 1.5, 0, 255).astype(np.uint8)
        filtered = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    elif emotion == "sad":
        # Escala de grises con tono azulado
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        blue_overlay = np.full_like(gray, (255, 0, 0))  # BGR: azul
        filtered = cv2.addWeighted(gray, 0.7, blue_overlay, 0.3, 0)

    elif emotion == "angry":
        # Tinte rojizo
        red_overlay = np.full_like(frame, (0, 0, 255))  # BGR: rojo
        filtered = cv2.addWeighted(frame, 0.6, red_overlay, 0.4, 0)

    elif emotion == "surprise":
        # Aumentar brillo y contraste
        filtered = cv2.convertScaleAbs(frame, alpha=1.3, beta=25)

    else:
        # Emociones neutras / desconocidas: filtro suave sin cambios fuertes
        filtered = cv2.GaussianBlur(frame, (7, 7), 0)

    return filtered

def run_emotion_filter_demo():
    cap = cv2.VideoCapture(1)
    if not cap.isOpened():
        raise RuntimeError("No se pudo abrir la webcam.")

    print("[INFO] Pulsar ESC para salir.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        try:
            # DeepFace devuelve una lista de dicts; usamos la primera cara
            obj = DeepFace.analyze(
                img_path=frame,
                actions=['emotion'],
                enforce_detection=True
            )
            # En versiones nuevas devuelve lista; en otras, dict
            if isinstance(obj, list):
                dominant_emotion = obj[0]['dominant_emotion']
            else:
                dominant_emotion = obj['dominant_emotion']

            filtered = apply_emotion_filter(frame, dominant_emotion)
            txt = f"Emocion: {dominant_emotion}"
            cv2.putText(filtered, txt, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        except Exception:
            # Si no detecta cara, mostramos el frame original con aviso
            filtered = frame.copy()
            cv2.putText(filtered, "Sin cara / no detectada", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        cv2.imshow("Prototipo 2 - Filtros por emocion", filtered)
        if cv2.waitKey(1) & 0xFF == 27:  # ESC
            break

    cap.release()
    cv2.destroyAllWindows()


# EJEMPLO DE USO:
run_emotion_filter_demo()



[INFO] Pulsar ESC para salir.


In [3]:
import cv2
import numpy as np
from deepface import DeepFace 
import pygame

def overlay_on_face(frame, region, overlay, scale=1.4):
    x = region["x"]
    y = region["y"]
    w = int(region["w"] * scale)
    h = int(region["h"] * scale)
    x = x - (w - region["w"]) // 2
    y = y - (h - region["h"]) // 2
    x = max(0, x)
    y = max(0, y)
    h_frame, w_frame = frame.shape[:2]
    w = min(w, w_frame - x)
    h = min(h, h_frame - y)
    if w <= 0 or h <= 0:
        return frame

    overlay_resized = cv2.resize(overlay, (w, h))
    if overlay_resized.shape[2] == 4:
        b, g, r, a = cv2.split(overlay_resized)
        overlay_bgr = cv2.merge((b, g, r))
        alpha = a.astype(float) / 255.0
        alpha = np.stack([alpha, alpha, alpha], axis=-1)
        roi = frame[y:y + h, x:x + w].astype(float)
        blended = alpha * overlay_bgr.astype(float) + (1 - alpha) * roi
        frame[y:y + h, x:x + w] = blended.astype(np.uint8)
    else:
        frame[y:y + h, x:x + w] = overlay_resized

    return frame

def run_face_filter_demo():
    pygame.mixer.init()
    pygame.mixer.music.load("./audio/torero.mp3")

    overlay = cv2.imread("./images/chayanne.webp", cv2.IMREAD_UNCHANGED)
    if overlay is None:
        raise RuntimeError("No se pudo cargar ./images/chayanne.webp")

    cap = cv2.VideoCapture(1)
    if not cap.isOpened():
        raise RuntimeError("No se pudo abrir la webcam.")

    sonido_activo = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        cara_detectada = False

        try:
            obj = DeepFace.analyze(
                img_path=frame,
                actions=["emotion"],
                enforce_detection=True
            )

            if isinstance(obj, list):
                regions = [item["region"] for item in obj]
            else:
                regions = [obj["region"]]

            cara_detectada = True

            filtered = frame.copy()
            for region in regions:
                filtered = overlay_on_face(filtered, region, overlay, scale=1.4)

        except Exception:
            filtered = frame.copy()

        if cara_detectada and not sonido_activo:
            pygame.mixer.music.play()
            sonido_activo = True
        elif not cara_detectada and sonido_activo:
            pygame.mixer.music.stop()
            sonido_activo = False

        cv2.imshow("Filtro Chayanne", filtered)
        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()
    pygame.mixer.quit()

run_face_filter_demo()


pygame 2.6.1 (SDL 2.28.4, Python 3.11.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
